[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/32_nlp/03_word_embeddings.ipynb)

# Bài 03 — Word embeddings

**🎯 Mục tiêu**
- Vì sao one-hot không đủ; **giả thuyết phân bố** (distributional hypothesis): nghĩa của từ thể hiện qua ngữ cảnh.
- Hai cách học vector từ từ văn bản **không nhãn**: đếm (ma trận đồng xuất hiện → **PPMI → SVD**) và dự đoán (**word2vec skip-gram + negative sampling**, cài bằng PyTorch).
- **Đánh giá** embedding: láng giềng gần nhất, từ đồng nghĩa, trực quan hoá.
- Dùng embedding cho **phân loại khi ít nhãn**; giới hạn của embedding tĩnh (đa nghĩa) → embedding ngữ cảnh.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/32_nlp

In [ ]:
import re
import time
import unicodedata
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from ml_course.datasets import NEWS_LEXICON, NEWS_SYNONYMS, make_news_vi
from ml_course.visualization import set_style

try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("Bài này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()
device = "cuda" if torch.cuda.is_available() else "cpu"
LEXICON = {tuple(w.lower().split()) for w in NEWS_LEXICON}
MAX_LEN = max(map(len, LEXICON))


def segment(text):                                                     # tách từ khớp cực đại (bài 01)
    syl = re.findall(r"\w[\w\-]*", unicodedata.normalize("NFC", text).lower())
    tokens, i = [], 0
    while i < len(syl):
        for L in range(min(MAX_LEN, len(syl) - i), 0, -1):
            if L == 1 or tuple(syl[i:i + L]) in LEXICON:
                tokens.append("_".join(syl[i:i + L]))
                i += L
                break
    return tokens


unlabeled = make_news_vi(20000, seed=1)                                # ngữ liệu KHÔNG dùng nhãn
docs = [segment(t) for t in unlabeled.text]
counts = Counter(w for d in docs for w in d)
vocab = sorted(w for w, c in counts.items() if c >= 3)
w2i = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
print(f"{len(docs):,} tin | {sum(counts.values()):,} token | từ vựng {V} từ (xuất hiện ≥ 3 lần)")
print(docs[0])

## 1. One-hot và giả thuyết phân bố

One-hot: mỗi từ là một vector đơn vị — mọi cặp từ **trực giao**, khoảng cách bằng nhau: "smartphone" cách "điện thoại thông minh" đúng bằng cách "sốt xuất huyết". Mô hình không thể chuyển hiểu biết từ từ này sang từ kia.

**Giả thuyết phân bố** (Harris 1954; Firth 1957: *"You shall know a word by the company it keeps"*): các từ xuất hiện trong ngữ cảnh giống nhau có nghĩa giống nhau. "smartphone" và "điện_thoại_thông_minh" cùng đi với "ra mắt", "camera", "Samsung" → nên có vector gần nhau. Ý tưởng này biến **văn bản không nhãn** thành tín hiệu học.

## 2. Cách đếm: đồng xuất hiện → PPMI → SVD

1. Ma trận đồng xuất hiện $C_{wc}$: số lần từ ngữ cảnh $c$ nằm trong cửa sổ ±3 quanh từ $w$.
2. Đếm thô bị chi phối bởi từ phổ biến → dùng **PMI** $= \log\frac{P(w, c)}{P(w)P(c)}$ (đồng xuất hiện nhiều hơn ngẫu nhiên bao nhiêu), cắt âm về 0 (**PPMI**).
3. **SVD** cắt cụt (chương 21) nén mỗi hàng thành vector 50 chiều — giống LSA.

In [ ]:
WINDOW = 3
centers, contexts = [], []
for d in docs:
    ids = [w2i[w] for w in d if w in w2i]
    for i, c in enumerate(ids):
        for j in range(max(0, i - WINDOW), min(len(ids), i + WINDOW + 1)):
            if j != i:
                centers.append(c)
                contexts.append(ids[j])
centers, contexts = np.array(centers), np.array(contexts)

C = np.zeros((V, V))
np.add.at(C, (centers, contexts), 1)
total = C.sum()
p_w, p_c = C.sum(1, keepdims=True) / total, C.sum(0, keepdims=True) / total
with np.errstate(divide="ignore"):
    ppmi = np.maximum(np.log(C / total / (p_w * p_c)), 0)
U, S, _ = np.linalg.svd(ppmi)
E_svd = U[:, :50] * np.sqrt(S[:50])
print(f"{len(centers):,} cặp (từ, ngữ cảnh) | ma trận PPMI {ppmi.shape}, {np.mean(ppmi > 0):.1%} khác 0 → embedding {E_svd.shape}")

## 3. Cách dự đoán: word2vec skip-gram với negative sampling

Skip-gram (Mikolov et al., 2013): với mỗi từ trung tâm $w$, dự đoán các từ ngữ cảnh $c$ xung quanh. Softmax trên cả từ vựng quá đắt → **negative sampling**: phân biệt cặp thật $(w, c)$ với $k$ cặp "giả" $(w, c')$, $c'$ lấy ngẫu nhiên theo tần suất$^{0.75}$:

$$\mathcal L = -\log\sigma(\mathbf u_c^\top\mathbf v_w) - \sum_{i=1}^{k}\log\sigma(-\mathbf u_{c'_i}^\top\mathbf v_w)$$

Mỗi từ có hai vector: $\mathbf v$ (khi là trung tâm) và $\mathbf u$ (khi là ngữ cảnh); embedding thường lấy $\mathbf v$. Levy & Goldberg (2014) chỉ ra skip-gram negative sampling ngầm phân tích **ma trận PMI dịch chuyển** — hai cách ở mục 2 và 3 liên hệ chặt chẽ.

In [ ]:
class SkipGramNS(nn.Module):
    def __init__(self, V, d=50):
        super().__init__()
        self.center, self.context = nn.Embedding(V, d), nn.Embedding(V, d)
        nn.init.uniform_(self.center.weight, -0.5 / d, 0.5 / d)
        nn.init.zeros_(self.context.weight)

    def forward(self, w, c, neg):
        v = self.center(w)                                             # (B, d)
        pos = nn.functional.logsigmoid((v * self.context(c)).sum(-1))
        negs = nn.functional.logsigmoid(-(self.context(neg) @ v.unsqueeze(-1)).squeeze(-1)).sum(-1)
        return -(pos + negs).mean()


noise = torch.tensor([counts[w] for w in vocab], dtype=torch.float) ** 0.75
noise /= noise.sum()
cen_t, ctx_t = torch.tensor(centers), torch.tensor(contexts)
torch.manual_seed(0)
sgns = SkipGramNS(V).to(device)
opt = torch.optim.Adam(sgns.parameters(), lr=3e-3)
g = torch.Generator().manual_seed(0)
t0 = time.perf_counter()
for epoch in range(3):
    perm = torch.randperm(len(cen_t), generator=g)
    for s in range(0, len(perm), 2048):
        idx = perm[s:s + 2048]
        neg = torch.multinomial(noise, len(idx) * 5, replacement=True, generator=g).view(len(idx), 5)
        opt.zero_grad()
        loss = sgns(cen_t[idx].to(device), ctx_t[idx].to(device), neg.to(device))
        loss.backward()
        opt.step()
    print(f"epoch {epoch + 1}: loss = {loss.item():.3f} | {time.perf_counter() - t0:.0f}s")
E_w2v = sgns.center.weight.detach().cpu().numpy()

## 4. Đánh giá embedding

### 4.1. Láng giềng gần nhất (cosine)

In [ ]:
def normalize(E):
    return E / np.linalg.norm(E, axis=1, keepdims=True)


def neighbors(E, word, k=5):
    En = normalize(E)
    sims = En @ En[w2i[word]]
    order = np.argsort(-sims)[1:k + 1]
    return [(vocab[i], round(float(sims[i]), 2)) for i in order]


for w in ["smartphone", "hlv", "vắc_xin", "điểm_chuẩn", "lãi_suất", "phú_quốc"]:
    print(f"{w:12s} word2vec: {[n for n, _ in neighbors(E_w2v, w)]}")
    print(f"{'':12s} PPMI-SVD: {[n for n, _ in neighbors(E_svd, w)]}")

Không ai cho mô hình biết chủ đề hay nghĩa của từ, nhưng láng giềng của "smartphone" là "điện_thoại_thông_minh", "tablet", "camera"; của "lãi_suất" là "tỷ_giá", "lợi_nhuận" — cấu trúc nghĩa nổi lên chỉ từ ngữ cảnh.

### 4.2. Từ đồng nghĩa

Bộ dữ liệu cho sẵn các nhóm đồng nghĩa (`NEWS_SYNONYMS`) — một **bộ đánh giá** độc lập với quá trình huấn luyện. Đo: cosine trung bình giữa các cặp đồng nghĩa so với cặp ngẫu nhiên, và tỉ lệ cặp mà từ kia nằm trong 5 láng giềng gần nhất.

In [ ]:
pairs = []
for group in NEWS_SYNONYMS:
    toks = [t for t in ("_".join(x.lower().split()) for x in group) if t in w2i]
    pairs += [(toks[i], toks[j]) for i in range(len(toks)) for j in range(i + 1, len(toks))]
rng = np.random.default_rng(0)
random_pairs = rng.integers(0, V, (2000, 2))
for name, E in [("word2vec", E_w2v), ("PPMI-SVD", E_svd), ("ngẫu nhiên", rng.normal(size=(V, 50)))]:
    En = normalize(E)
    syn = np.mean([En[w2i[a]] @ En[w2i[b]] for a, b in pairs])
    rand = np.mean([En[i] @ En[j] for i, j in random_pairs if i != j])
    hit = np.mean([b in [n for n, _ in neighbors(E, a)] or a in [n for n, _ in neighbors(E, b)] for a, b in pairs])
    print(f"{name:10s}: cosine cặp đồng nghĩa = {syn:.2f} | cặp ngẫu nhiên = {rand:.2f} | đồng nghĩa trong top-5 = {hit:.0%} ({len(pairs)} cặp)")

Cả hai phương pháp đặt hơn 90% cặp đồng nghĩa vào top-5 láng giềng (embedding ngẫu nhiên: chỉ vài %, đúng mức may rủi). Lưu ý: cosine tuyệt đối không so được giữa hai phương pháp (word2vec có mọi vector hơi "cùng hướng" — cặp ngẫu nhiên cũng có cosine dương); hãy so **thứ hạng** hoặc chênh lệch với cặp ngẫu nhiên.

### 4.3. Trực quan hoá

In [ ]:
labeled_all = make_news_vi(4000, seed=0)
topic_of = {}
for text, topic in zip(labeled_all.text, labeled_all.topic):          # chủ đề "chính" của từ = chủ đề nó xuất hiện nhiều nhất
    for w in set(segment(text)):
        topic_of.setdefault(w, Counter())[topic] += 1
show = [w for w in vocab if w in topic_of and "_" in w and counts[w] >= 50
        and topic_of[w].most_common(1)[0][1] / sum(topic_of[w].values()) > 0.7]
P = PCA(2).fit_transform(normalize(E_w2v)[[w2i[w] for w in show]])
fig, ax = plt.subplots(figsize=(9, 6.5))
topics = sorted({topic_of[w].most_common(1)[0][0] for w in show})
for k, t in enumerate(topics):
    idx = [i for i, w in enumerate(show) if topic_of[w].most_common(1)[0][0] == t]
    ax.scatter(P[idx, 0], P[idx, 1], s=18, color=f"C{k}", label=t)
for i in range(0, len(show), 3):
    ax.annotate(show[i], P[i], fontsize=7, alpha=0.8)
ax.legend(fontsize=8)
ax.set_title("word2vec (PCA 2D) — mỗi điểm một từ, màu = chủ đề chính")
plt.tight_layout()
plt.show()

## 5. Embedding cho phân loại khi ít nhãn

Biểu diễn mỗi văn bản bằng **trung bình** embedding các từ, rồi Logistic Regression — so với TF-IDF (bài 02) khi chỉ có 30, 100, 300 văn bản có nhãn (cùng tập train/test như bài 02, 5 lần chọn mẫu). Embedding được học từ 20 000 tin **không nhãn** — rẻ, dễ thu thập.

In [ ]:
train_idx, test_idx = train_test_split(np.arange(len(labeled_all)), test_size=0.3, random_state=0, stratify=labeled_all.topic)
labeled_docs = [segment(t) for t in labeled_all.text]
y = labeled_all.topic.values


def doc_vector(E, tokens):
    ids = [w2i[w] for w in tokens if w in w2i]
    return normalize(E)[ids].mean(0) if ids else np.zeros(E.shape[1])


features = {"word2vec (trung bình)": np.array([doc_vector(E_w2v, d) for d in labeled_docs]),
            "PPMI-SVD (trung bình)": np.array([doc_vector(E_svd, d) for d in labeled_docs])}
joined = np.array([" ".join(d) for d in labeled_docs])
for n in [30, 100, 300, len(train_idx)]:
    res = {"TF-IDF": [], **{k: [] for k in features}}
    for seed in range(5 if n < len(train_idx) else 1):
        idx = np.random.default_rng(seed).choice(train_idx, n, replace=False) if n < len(train_idx) else train_idx
        vec = TfidfVectorizer(token_pattern=r"\S+").fit(joined[idx])
        lr = LogisticRegression(max_iter=3000, C=10).fit(vec.transform(joined[idx]), y[idx])
        res["TF-IDF"].append((lr.predict(vec.transform(joined[test_idx])) == y[test_idx]).mean())
        for k, X in features.items():
            lr = LogisticRegression(max_iter=3000, C=10).fit(X[idx], y[idx])
            res[k].append((lr.predict(X[test_idx]) == y[test_idx]).mean())
    print(f"{n:5d} nhãn: " + " | ".join(f"{k} = {np.mean(v):.3f}" for k, v in res.items()))

- Với **30 nhãn**, embedding vượt TF-IDF rõ rệt: một văn bản test chứa "smartphone" vẫn được hiểu dù 30 văn bản train chỉ chứa "điện thoại thông minh" — thông tin này đến từ ngữ liệu không nhãn. Đây là dạng đơn giản của **học bán giám sát / tiền huấn luyện**.
- Khi đủ nhãn, các cách hội tụ về cùng mức (trần do nhãn nhiễu và câu lạc đề). Trung bình vector làm mất thứ tự và chi tiết, nên với nhiều nhãn TF-IDF + n-gram vẫn là lựa chọn rất tốt.

## 6. Giới hạn của embedding tĩnh

- **Một vector cho mỗi từ, bất kể ngữ cảnh**: "thị_trường" dùng trong kinh tế (bất động sản), công nghệ (điện thoại) và du lịch → vector của nó là "trung bình" các nghĩa. Tương tự "đường" (con đường / đường ăn), "bàn" (cái bàn / bàn thắng / bàn bạc).
- **Thiên lệch xã hội**: embedding học từ văn bản thật mang theo định kiến của dữ liệu (giới, nghề nghiệp — Bolukbasi et al., 2016).
- **Từ chưa gặp**: không có vector → fastText dùng n-gram ký tự để ghép vector cho từ mới.

In [ ]:
for w in ["thị_trường", "ứng_dụng_di_động", "giảm_giá"]:
    dist = topic_of.get(w, Counter())
    print(f"{w:16s} xuất hiện theo chủ đề: {dict(dist.most_common(3))}")
    print(f"{'':16s} láng giềng: {[n for n, _ in neighbors(E_w2v, w, 6)]}")

Láng giềng của từ đa ngữ cảnh là hỗn hợp từ nhiều chủ đề. **Embedding ngữ cảnh** (ELMo, BERT — chương 31) giải quyết bằng cách tính vector của từ **từ cả câu**: "thị trường" trong câu về chung cư và trong câu về smartphone nhận hai vector khác nhau.

Trong thực tế, dùng embedding **tiền huấn luyện** trên ngữ liệu lớn thay vì tự train: fastText cho tiếng Việt (157 ngôn ngữ, Grave et al., 2018), PhoW2V; hoặc bỏ qua embedding tĩnh và dùng mô hình Transformer tiền huấn luyện (PhoBERT, sentence embedding — bài 04).

## 🧠 Kiểm tra nhanh

1. Vì sao one-hot không biểu diễn được độ tương đồng giữa từ?
2. Phát biểu giả thuyết phân bố. Nó biến dữ liệu không nhãn thành tín hiệu học như thế nào?
3. PPMI là gì? Vì sao không dùng đếm đồng xuất hiện thô?
4. Mô tả skip-gram với negative sampling. Phân phối nhiễu lấy theo tần suất$^{0.75}$ để làm gì?
5. Vì sao embedding giúp nhiều khi ít nhãn nhưng ít khi nhiều nhãn?
6. Giới hạn của embedding tĩnh? Embedding ngữ cảnh khắc phục ra sao?

## 📌 Tóm tắt

- Embedding: vector dày, các từ có ngữ cảnh giống nhau thì gần nhau — học từ văn bản không nhãn.
- Đếm (PPMI + SVD) và dự đoán (word2vec SGNS) cho kết quả tương tự và liên hệ toán học chặt chẽ.
- Đánh giá bằng láng giềng, bộ từ đồng nghĩa/tương đồng; cosine chỉ so được trong cùng một không gian.
- Trung bình embedding + LogReg vượt TF-IDF khi rất ít nhãn; embedding tĩnh không xử lý đa nghĩa → BERT.

## 📚 Tài liệu tham khảo

- Mikolov et al. (2013) — *Distributed Representations of Words and Phrases and their Compositionality*
- Levy & Goldberg (2014) — *Neural Word Embedding as Implicit Matrix Factorization*
- Pennington, Socher & Manning (2014) — *GloVe: Global Vectors for Word Representation*
- Bojanowski et al. (2017) — *Enriching Word Vectors with Subword Information* (fastText)
- Bolukbasi et al. (2016) — *Man is to Computer Programmer as Woman is to Homemaker? Debiasing Word Embeddings*
- Jurafsky & Martin — *Speech and Language Processing*, chương 6 (Vector Semantics and Embeddings)